# Netflix Movie Recommendation System Pipeline
### Complete Beginner-Level Data Analysis, EDA, Feature Engineering & Pipeline

This notebook implements the complete 28-step pipeline for analyzing `netflix_data.csv` and preparing features for both Content-Based and Collaborative Filtering Movie Recommendation Systems.


## Step 1: Import Required Libraries
Import essential Python libraries for data manipulation, numerical calculations, visualization, statistical analysis, and machine learning feature extraction.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import re

# Display helper for notebooks and scripts
try:
    from IPython.display import display
except ImportError:
    display = print

# Scikit-learn modules for feature engineering & scaling
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler

# Visualization settings
%matplotlib inline
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
print("Libraries imported successfully!")


## Step 2: Load Dataset
Load the Netflix dataset `netflix_data.csv` using Pandas `read_csv()` and explore the initial structure with `head()`, `tail()`, `sample()`, `shape`, and `columns`.


In [ ]:
# Load Movie/Netflix Dataset
df = pd.read_csv('netflix_data.csv')

print("--- Dataset Loaded Successfully ---")
print(f"Dataset Shape: {df.shape} (Rows: {df.shape[0]}, Columns: {df.shape[1]})")

# Display first 5 rows
print("\n--- Head (First 5 Rows) ---")
display(df.head())

# Display last 5 rows
print("\n--- Tail (Last 5 Rows) ---")
display(df.tail())

# Display random sample of 5 rows
print("\n--- Sample (5 Random Rows) ---")
display(df.sample(5, random_state=42))

# List all columns
print("\n--- Columns List ---")
print(df.columns.tolist())


## Step 3: Understand Dataset
Inspect dataset structure, summary stats, data types, and unique value counts using `info()`, `describe()`, `dtypes`, `nunique()`, `unique()`, and `value_counts()`.


In [ ]:
print("--- Dataset Info ---")
df.info()

print("\n--- Data Types ---")
print(df.dtypes)

print("\n--- Summary Statistics (Numerical & Categorical) ---")
display(df.describe(include='all'))

print("\n--- Number of Unique Values per Column ---")
print(df.nunique())

print("\n--- Unique Types of Content ---")
print(df['type'].unique())

print("\n--- Content Type Value Counts ---")
print(df['type'].value_counts())

print("\n--- Top 10 Rating Categories Value Counts ---")
print(df['rating'].value_counts().head(10))


## Step 4: Initial EDA
Perform basic dataset inspection: row count, column count, data types, missing values (`isnull().sum()`), duplicate records (`duplicated().sum()`), unique titles, genres, and rating ranges.


In [ ]:
# Check dimensions
num_rows, num_cols = df.shape
print(f"Number of Rows: {num_rows}")
print(f"Number of Columns: {num_cols}")

# Check missing values count
print("\n--- Missing Values Count ---")
print(df.isnull().sum())

# Check duplicate records
num_duplicates = df.duplicated().sum()
print(f"\nNumber of Duplicate Records: {num_duplicates}")

# Unique Titles & Genres
print(f"Unique Titles: {df['title'].nunique()}")
print(f"Unique Rating Categories: {df['rating'].nunique()}")
print(f"Unique Directors: {df['director'].nunique()}")
print(f"Unique Countries: {df['country'].nunique()}")

# Basic summary of release year
print(f"\nRelease Year Range: {df['release_year'].min()} to {df['release_year'].max()}")
print(f"Mean Release Year: {df['release_year'].mean():.2f}")
print(f"Median Release Year: {df['release_year'].median()}")


## Step 5: Missing Value Analysis
Identify missing values, calculate percentage of missingness, visualize missing values using a heatmap/barplot, and decide appropriate handling strategies (fill categorical text with "Unknown", mode, etc.).


In [ ]:
# Calculate missing value count and percentage
missing_count = df.isnull().sum()
missing_percent = (missing_count / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing Count': missing_count,
    'Percentage (%)': missing_percent
}).sort_values(by='Missing Count', ascending=False)

print("--- Missing Values Summary ---")
display(missing_df[missing_df['Missing Count'] > 0])

# Visualize missing values heatmap
plt.figure(figsize=(12, 6))
sns.heatmap(df.isnull(), cbar=False, cmap='viridis', yticklabels=False)
plt.title('Missing Value Heatmap in Netflix Dataset', fontsize=14)
plt.show()

# Imputation Decision Strategy:
# - 'director', 'cast', 'country': Fill missing text values with "Unknown"
# - 'date_added': Fill missing with mode
# - 'rating': Fill missing with mode

df['director'] = df['director'].fillna('Unknown')
df['cast'] = df['cast'].fillna('Unknown')
df['country'] = df['country'].fillna('Unknown')
df['date_added'] = df['date_added'].fillna(df['date_added'].mode()[0])
df['rating'] = df['rating'].fillna(df['rating'].mode()[0])

print("Missing values after imputation:")
print(df.isnull().sum())


## Step 6: Duplicate Data Analysis
Find and display any duplicate records in the dataset, count them, and drop duplicates using `drop_duplicates()`.


In [ ]:
# Check duplicate rows
duplicates = df[df.duplicated()]
print(f"Duplicate Rows Count: {len(duplicates)}")

if len(duplicates) > 0:
    print("Displaying Duplicate Rows:")
    display(duplicates)
    df = df.drop_duplicates()
    print("Duplicates removed using drop_duplicates().")
else:
    print("No duplicate rows found in dataset.")

print(f"Dataset shape after duplicate check: {df.shape}")


## Step 7: Data Type Correction
Convert duration into numerical values (minutes for Movies, seasons for TV Shows), parse `date_added` into `datetime`, and convert categorical columns into category dtype where appropriate.


In [ ]:
# Convert date_added to datetime format
df['date_added_clean'] = df['date_added'].str.strip()
df['date_added_dt'] = pd.to_datetime(df['date_added_clean'], errors='coerce')
df['added_year'] = df['date_added_dt'].dt.year

# Parse duration into numerical column (duration_num)
df['duration_num'] = df['duration'].str.extract('(\d+)').astype(float)

# Separate movie duration (minutes)
df['duration_min'] = np.where(df['type'] == 'Movie', df['duration_num'], np.nan)
df['seasons_count'] = np.where(df['type'] == 'TV Show', df['duration_num'], np.nan)

# Convert categorical columns
df['type'] = df['type'].astype('category')
df['rating'] = df['rating'].astype('category')

print("Data types after conversion:")
print(df[['type', 'rating', 'release_year', 'added_year', 'duration_min', 'seasons_count']].dtypes)


## Step 8: Basic Data Cleaning
Clean text fields: strip extra spaces, convert text to lowercase, rename/clean columns, standardize genre names, and re-check `info()` and `describe()`.


In [ ]:
# Clean text columns: remove trailing/leading spaces and lowercase text
text_cols = ['title', 'director', 'cast', 'country', 'listed_in', 'description']
for col in text_cols:
    df[col + '_clean'] = df[col].astype(str).str.strip().str.lower()

# Standardize listed_in genre column
df['primary_genre'] = df['listed_in_clean'].apply(lambda x: x.split(',')[0].strip())

print("--- Cleaned Dataset Info ---")
df.info()

print("\n--- Cleaned Dataset Describe ---")
display(df.describe(include='all'))


## Step 9: Univariate EDA
Analyze columns individually.
- **Numerical columns**: Mean, Median, Mode, Min, Max, Standard Deviation, Variance, Quartiles.
- **Categorical columns**: `value_counts()`, `nunique()`, most common genre/movie/rating.
- **Plots**: Histogram, Boxplot, Bar chart, Count plot, Pie chart.


In [ ]:
# Numerical Column Analysis: duration_min for Movies
dur = df['duration_min'].dropna()

print("=== Numerical Univariate Stats for Movie Duration (minutes) ===")
print(f"Mean: {dur.mean():.2f}")
print(f"Median: {dur.median():.2f}")
print(f"Mode: {dur.mode()[0]:.2f}")
print(f"Min: {dur.min():.2f}")
print(f"Max: {dur.max():.2f}")
print(f"Std Dev: {dur.std():.2f}")
print(f"Variance: {dur.var():.2f}")
print(f"25th Quartile (Q1): {dur.quantile(0.25):.2f}")
print(f"75th Quartile (Q3): {dur.quantile(0.75):.2f}")

# Visualizations: Univariate
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. Histogram of Movie Duration
sns.histplot(dur, kde=True, ax=axes[0, 0], color='skyblue')
axes[0, 0].set_title('Histogram of Movie Duration')

# 2. Boxplot of Movie Duration
sns.boxplot(x=dur, ax=axes[0, 1], color='lightgreen')
axes[0, 1].set_title('Boxplot of Movie Duration')

# 3. Bar Chart of Content Type
type_counts = df['type'].value_counts()
axes[0, 2].bar(type_counts.index, type_counts.values, color=['#e50914', '#221f1f'])
axes[0, 2].set_title('Content Type Distribution (Movie vs TV Show)')

# 4. Count plot of Ratings
sns.countplot(data=df, y='rating', order=df['rating'].value_counts().head(10).index, ax=axes[1, 0], palette='viridis')
axes[1, 0].set_title('Top 10 Ratings Countplot')

# 5. Pie Chart of Content Type
axes[1, 1].pie(type_counts, labels=type_counts.index, autopct='%1.1f%%', colors=['#e50914', '#333333'], startangle=140)
axes[1, 1].set_title('Pie Chart of Content Types')

# 6. Bar Chart of Top 10 Primary Genres
top_genres = df['primary_genre'].value_counts().head(10)
sns.barplot(x=top_genres.values, y=top_genres.index, ax=axes[1, 2], palette='mako')
axes[1, 2].set_title('Top 10 Primary Genres')

plt.tight_layout()
plt.show()


## Step 10: Bivariate EDA
Analyze pairs of variables together.
- Release Year vs Duration
- Type vs Release Year
- Duration vs Rating
- Plots: Scatter plot, Line plot, Bar plot, Boxplot, Violin plot.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. Scatter Plot: Release Year vs Movie Duration
sns.scatterplot(data=df[df['type']=='Movie'], x='release_year', y='duration_min', alpha=0.5, ax=axes[0, 0], color='purple')
axes[0, 0].set_title('Scatter Plot: Release Year vs Movie Duration')

# 2. Line Plot: Number of Releases over Years by Type
year_type = df.groupby(['release_year', 'type'], observed=False).size().reset_index(name='count')
sns.lineplot(data=year_type[year_type['release_year']>=1990], x='release_year', y='count', hue='type', ax=axes[0, 1], palette=['red', 'black'])
axes[0, 1].set_title('Line Plot: Content Releases per Year (1990-2021)')

# 3. Bar Plot: Average Duration by Content Type
df_grouped_type = df.groupby('type', observed=False)['duration_num'].mean().reset_index()
sns.barplot(data=df_grouped_type, x='type', y='duration_num', ax=axes[0, 2], palette='Set2')
axes[0, 2].set_title('Bar Plot: Average Duration Units by Type')

# 4. Boxplot: Movie Duration across Ratings
top_ratings_list = df['rating'].value_counts().head(6).index
sns.boxplot(data=df[(df['type']=='Movie') & (df['rating'].isin(top_ratings_list))], x='rating', y='duration_min', ax=axes[1, 0], palette='Set3')
axes[1, 0].set_title('Boxplot: Movie Duration vs Rating')

# 5. Violin Plot: Movie Duration across Content Types
sns.violinplot(data=df[df['type']=='Movie'], x='type', y='duration_min', ax=axes[1, 1], color='salmon')
axes[1, 1].set_title('Violin Plot: Movie Duration Distribution')

# 6. Bar Plot: Top 5 Countries Content Output
top_countries = df[df['country']!='Unknown']['country'].value_counts().head(5)
sns.barplot(x=top_countries.values, y=top_countries.index, ax=axes[1, 2], palette='flare')
axes[1, 2].set_title('Bar Plot: Top 5 Content Producing Countries')

plt.tight_layout()
plt.show()


## Step 11: Multivariate EDA
Analyze relationships across multiple numeric features (e.g. `release_year`, `added_year`, `duration_min`, `seasons_count`).
Use `corr()`, `sns.heatmap()`, and `sns.pairplot()`.


In [ ]:
# Select numerical features for correlation analysis
num_features = df[['release_year', 'added_year', 'duration_min', 'seasons_count']].dropna()

# Correlation Matrix
corr_matrix = num_features.corr()
print("--- Correlation Matrix ---")
display(corr_matrix)

# Heatmap Visualization
plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('Multivariate Correlation Heatmap', fontsize=14)
plt.show()

# Pair Plot
sns.pairplot(df[['release_year', 'added_year', 'duration_min', 'type']].dropna(), hue='type', palette='Set1')
plt.suptitle('Multivariate Pair Plot', y=1.02)
plt.show()


## Step 12: Outlier Analysis — IQR Method
Detect outliers in numerical variables (e.g. `duration_min`, `release_year`) using the Interquartile Range (IQR) method.
$$IQR = Q3 - Q1$$
$$\text{Lower Limit} = Q1 - 1.5 \times IQR$$
$$\text{Upper Limit} = Q3 + 1.5 \times IQR$$


In [ ]:
# IQR Method for Movie Duration
movie_dur = df[df['type']=='Movie']['duration_min'].dropna()

Q1 = movie_dur.quantile(0.25)
Q3 = movie_dur.quantile(0.75)
IQR = Q3 - Q1

lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

outliers_iqr = movie_dur[(movie_dur < lower_limit) | (movie_dur > upper_limit)]

print("=== Outlier Analysis using IQR Method ===")
print(f"Q1 (25th percentile): {Q1:.2f}")
print(f"Q3 (75th percentile): {Q3:.2f}")
print(f"IQR: {IQR:.2f}")
print(f"Lower Limit: {lower_limit:.2f}")
print(f"Upper Limit: {upper_limit:.2f}")
print(f"Number of Outliers Detected (IQR): {len(outliers_iqr)}")
print(f"Percentage of Outliers (IQR): {(len(outliers_iqr) / len(movie_dur)) * 100:.2f}%")


## Step 13: Outlier Analysis — Z-Score Method
Detect outliers in numerical features using the Z-Score method (`scipy.stats.zscore`).
$$Z = \frac{X - \text{Mean}}{\text{Std Dev}}$$
Identify values where $|Z| > 3$.


In [ ]:
# Z-Score Method for Movie Duration
mean_dur = movie_dur.mean()
std_dur = movie_dur.std()

z_scores = stats.zscore(movie_dur)
outliers_zscore = movie_dur[np.abs(z_scores) > 3]

print("=== Outlier Analysis using Z-Score Method ===")
print(f"Mean: {mean_dur:.2f}")
print(f"Standard Deviation: {std_dur:.2f}")
print(f"Number of Outliers Detected (|Z| > 3): {len(outliers_zscore)}")
print(f"Percentage of Outliers (Z-Score): {(len(outliers_zscore) / len(movie_dur)) * 100:.2f}%")


## Step 14: Compare IQR and Z-Score
Compare results obtained from IQR and Z-Score outlier detection methods.


In [ ]:
comparison_df = pd.DataFrame({
    'Metric': ['Method Criteria', 'Lower Bound', 'Upper Bound', 'Outliers Detected Count', 'Outliers Percentage'],
    'IQR Method': ['Q1 - 1.5*IQR / Q3 + 1.5*IQR', f"{lower_limit:.2f}", f"{upper_limit:.2f}", len(outliers_iqr), f"{(len(outliers_iqr)/len(movie_dur))*100:.2f}%"],
    'Z-Score Method': ['|Z| > 3', f"{mean_dur - 3*std_dur:.2f}", f"{mean_dur + 3*std_dur:.2f}", len(outliers_zscore), f"{(len(outliers_zscore)/len(movie_dur))*100:.2f}%"]
})

print("--- Outlier Detection Method Comparison Table ---")
display(comparison_df)

print("\nDecision Rationale:")
print("- IQR method is non-parametric and detects moderate extremes (short films < 45m or long films > 150m).")
print("- Z-Score is stricter (|Z|>3) and identifies extreme values.")
print("- For recommendation systems, genuine movie durations (e.g. short films or long epics) should be retained as valid signal rather than removed.")


## Step 15: Outlier Visualization
Visualize distribution of movie duration before and after capping/filtering extreme outliers.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Before outlier removal boxplot
sns.boxplot(x=movie_dur, ax=axes[0], color='lightcoral')
axes[0].set_title(f'Before Outlier Removal (Total: {len(movie_dur)})')

# After IQR outlier filter visualization
movie_dur_cleaned = movie_dur[(movie_dur >= lower_limit) & (movie_dur <= upper_limit)]
sns.boxplot(x=movie_dur_cleaned, ax=axes[1], color='lightgreen')
axes[1].set_title(f'After IQR Outlier Removal (Total: {len(movie_dur_cleaned)})')

plt.tight_layout()
plt.show()


## Step 16: Deep EDA — Movie Data
Deep dive into movie content metadata: top-rated content categories, common genres, content volume by year, average duration per genre.


In [ ]:
movies_df = df[df['type'] == 'Movie'].copy()

print("--- Movie Deep EDA Statistics ---")
print(f"Total Movies: {len(movies_df)}")
print(f"Average Movie Duration: {movies_df['duration_min'].mean():.2f} minutes")

# Top Genres for Movies
top_movie_genres = movies_df['primary_genre'].value_counts().head(10)

# Plots for Deep Movie EDA
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. Top 10 Movie Genres
sns.barplot(x=top_movie_genres.values, y=top_movie_genres.index, ax=axes[0, 0], palette='Blues_r')
axes[0, 0].set_title('Top 10 Movie Primary Genres')

# 2. Movie Duration Distribution Histogram
sns.histplot(movies_df['duration_min'], bins=30, kde=True, ax=axes[0, 1], color='crimson')
axes[0, 1].set_title('Movie Duration Distribution')

# 3. Movies Produced per Year (Post 2000)
movies_by_year = movies_df[movies_df['release_year'] >= 2000]['release_year'].value_counts().sort_index()
axes[1, 0].plot(movies_by_year.index, movies_by_year.values, marker='o', color='darkorange')
axes[1, 0].set_title('Number of Movies Released per Year (2000-2021)')
axes[1, 0].set_xlabel('Release Year')

# 4. Average Movie Duration by Top 8 Genres
avg_dur_genre = movies_df.groupby('primary_genre')['duration_min'].mean().loc[top_movie_genres.index[:8]]
sns.barplot(x=avg_dur_genre.values, y=avg_dur_genre.index, ax=axes[1, 1], palette='Purples_r')
axes[1, 1].set_title('Average Movie Duration by Top Genres')

plt.tight_layout()
plt.show()


## Step 17: Deep EDA — User Rating Data & Simulation
Analyze rating distribution across metadata. Since `netflix_data.csv` provides content metadata, we simulate a reproducible sample User-Rating Interaction dataset ($N=5,000$ ratings across 500 users) over Netflix items to demonstrate user-based collaborative filtering analytics.


In [ ]:
np.random.seed(42)

# Generate synthetic user interaction ratings dataset for Collaborative Filtering analysis
num_ratings = 5000
user_ids = [f"User_{i}" for i in np.random.randint(1, 201, size=num_ratings)]
show_ids = np.random.choice(df['show_id'], size=num_ratings)
ratings_list = np.random.choice([1.0, 2.0, 3.0, 4.0, 5.0], size=num_ratings, p=[0.05, 0.10, 0.25, 0.35, 0.25])

ratings_df = pd.DataFrame({
    'user_id': user_ids,
    'show_id': show_ids,
    'user_rating': ratings_list
})

print("--- Simulated User Rating Interaction Dataset ---")
print(f"Total Synthetic Ratings: {len(ratings_df)}")
print(f"Unique Users: {ratings_df['user_id'].nunique()}")
print(f"Unique Items Rated: {ratings_df['show_id'].nunique()}")
display(ratings_df.head())

# User rating EDA
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Rating score distribution
sns.countplot(data=ratings_df, x='user_rating', ax=axes[0], palette='viridis')
axes[0].set_title('Distribution of User Ratings (1-5 Stars)')

# Top 10 most active users
top_active_users = ratings_df['user_id'].value_counts().head(10)
sns.barplot(x=top_active_users.values, y=top_active_users.index, ax=axes[1], palette='magma')
axes[1].set_title('Top 10 Most Active Users (Rating Count)')

plt.tight_layout()
plt.show()


## Step 18: GroupBy Analysis
Perform multi-dimensional grouping using `groupby()` to analyze metrics (`mean`, `median`, `count`, `sum`, `min`, `max`, `std`).


In [ ]:
# Group by Content Type
print("=== GroupBy Content Type (Duration Analysis) ===")
type_grouped = df.groupby('type', observed=False)['duration_num'].agg(['mean', 'median', 'count', 'min', 'max', 'std'])
display(type_grouped)

# Group by Rating Category
print("\n=== GroupBy Rating Category (Movie Duration Stats) ===")
rating_grouped = df[df['type']=='Movie'].groupby('rating', observed=False)['duration_min'].agg(['mean', 'median', 'count', 'std']).head(8)
display(rating_grouped)

# Group by Primary Genre
print("\n=== GroupBy Primary Genre (Content Count & Release Year) ===")
genre_grouped = df.groupby('primary_genre')['release_year'].agg(['count', 'min', 'max', 'mean']).sort_values(by='count', ascending=False).head(10)
display(genre_grouped)


## Step 19: Aggregation
Use `agg()` to perform multiple statistics across multiple features simultaneously.


In [ ]:
# Advanced Multi-Column Aggregation
agg_summary = df.groupby('type', observed=False).agg({
    'release_year': ['min', 'max', 'mean'],
    'duration_num': ['mean', 'median', 'std'],
    'show_id': ['count']
})

print("--- Multi-Column Aggregation Summary ---")
display(agg_summary)


## Step 20: Correlation Analysis
Analyze relationships among numerical columns using `corr()` and plot a correlation heatmap table.


In [ ]:
# Compute correlation matrix on available numerical variables
corr_df = df[['release_year', 'added_year', 'duration_num', 'duration_min']].corr()

print("--- Numerical Correlation Matrix Table ---")
display(corr_df)

plt.figure(figsize=(7, 5))
sns.heatmap(corr_df, annot=True, cmap='YlGnBu', fmt='.3f', square=True)
plt.title('Correlation Heatmap Table', fontsize=13)
plt.show()


## Step 21: Content-Based Feature Engineering
Prepare textual features for Content-Based Recommender System:
1. Select text fields (`primary_genre`, `cast`, `director`, `description`).
2. Clean text, lowercase, remove special characters.
3. Combine features into a single string `combined_features`.
4. Vectorize text using `TfidfVectorizer()` and `CountVectorizer()`.


In [ ]:
# 1. Clean & combine textual metadata into combined_features column
df['combined_features'] = (
    df['primary_genre'] + " " +
    df['director_clean'] + " " +
    df['cast_clean'] + " " +
    df['description_clean']
)

print("Sample Combined Feature String for Recommender System:")
print(df['combined_features'].iloc[0][:150] + "...")

# 2. Apply TF-IDF Vectorizer
tfidf = TfidfVectorizer(stop_words='english', max_features=5000)
tfidf_matrix = tfidf.fit_transform(df['combined_features'])

print(f"\nTF-IDF Feature Matrix Shape: {tfidf_matrix.shape}")

# 3. Apply Count Vectorizer
count_vec = CountVectorizer(stop_words='english', max_features=5000)
count_matrix = count_vec.fit_transform(df['combined_features'])

print(f"CountVectorizer Feature Matrix Shape: {count_matrix.shape}")


## Step 22: Collaborative Filtering Feature Engineering
Construct User-Movie Matrix using `pivot_table()`, compute average user ratings, average item ratings, and interaction counts per user/item.


In [ ]:
# Create User-Item Interaction Matrix using pivot_table()
user_movie_matrix = ratings_df.pivot_table(
    index='user_id',
    columns='show_id',
    values='user_rating'
)

print(f"User-Movie Matrix Shape: {user_movie_matrix.shape}")

# Calculate User and Movie Summary Metrics
user_stats = ratings_df.groupby('user_id')['user_rating'].agg(
    avg_user_rating='mean',
    ratings_per_user='count'
)

movie_stats = ratings_df.groupby('show_id')['user_rating'].agg(
    avg_movie_rating='mean',
    ratings_per_movie='count'
)

print("\n--- Sample User Interaction Stats ---")
display(user_stats.head())

print("\n--- Sample Movie Rating Stats ---")
display(movie_stats.head())


## Step 23: Encoding
Convert categorical features into numerical form using One-Hot Encoding (`pd.get_dummies()`) and Label Encoding (`LabelEncoder()`).


In [ ]:
# 1. One-Hot Encoding on 'type' and 'rating'
df_encoded = pd.get_dummies(df, columns=['type'], prefix='type', drop_first=True)

print("One-Hot Encoded Type Columns:")
print([col for col in df_encoded.columns if col.startswith('type_')])

# 2. Label Encoding on 'rating' category
le = LabelEncoder()
df['rating_encoded'] = le.fit_transform(df['rating'].astype(str))

print("\nLabel Encoded Rating Sample:")
display(df[['rating', 'rating_encoded']].head())


## Step 24: Text Feature Engineering
Perform tokenization, stop-word removal, and extract feature names from TF-IDF and Count Vectorizers.


In [ ]:
# Extract top feature terms from TF-IDF
tfidf_feature_names = tfidf.get_feature_names_out()
print(f"Total Extracted Text Features: {len(tfidf_feature_names)}")
print("Sample Extracted Text Tokens/Terms:", tfidf_feature_names[500:515])


## Step 25: Scaling / Normalization
Scale numerical features (`release_year`, `duration_num`, `added_year`) using `StandardScaler` and `MinMaxScaler`.


In [ ]:
scaler_std = StandardScaler()
scaler_minmax = MinMaxScaler()

num_cols_to_scale = ['release_year', 'duration_num']
df_num_filled = df[num_cols_to_scale].fillna(df[num_cols_to_scale].median())

# Standard Scaling (Z-score normalization)
scaled_std = scaler_std.fit_transform(df_num_filled)
df_scaled_std = pd.DataFrame(scaled_std, columns=[c+'_std' for c in num_cols_to_scale])

# Min-Max Scaling (0 to 1 range)
scaled_minmax = scaler_minmax.fit_transform(df_num_filled)
df_scaled_minmax = pd.DataFrame(scaled_minmax, columns=[c+'_minmax' for c in num_cols_to_scale])

print("--- Scaled Features Sample ---")
display(pd.concat([df_num_filled.head(), df_scaled_std.head(), df_scaled_minmax.head()], axis=1))


## Step 26: Final Feature Dataset
Verify final dataset readiness prior to recommender model training.
- Zero missing values in key columns.
- Correct dtypes.
- Encoded categorical features.
- Vectorized text matrices.
- Scaled numerical features.


In [ ]:
print("=== Final Feature Verification Check ===")
print(f"1. Main Dataset Rows & Columns: {df.shape}")
print(f"2. Remaining Missing Values: {df[['title', 'combined_features', 'type']].isnull().sum().sum()}")
print(f"3. TF-IDF Text Feature Matrix Dimensions: {tfidf_matrix.shape}")
print(f"4. User-Movie Matrix Dimensions: {user_movie_matrix.shape}")
print(f"5. Scaled Numerical Features Ready: {df_scaled_std.shape[1]} columns")
print("Status: All datasets and feature representations are clean and ready for modeling!")


## Step 27: Final Visualization Before Modeling
Create a comprehensive visual summary of key distributions and features prior to recommendation model execution.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. Content Type Breakdown
df['type'].value_counts().plot(kind='bar', ax=axes[0, 0], color=['#e50914', '#221f1f'])
axes[0, 0].set_title('Content Type Breakdown')

# 2. Movie Duration Histogram
sns.histplot(df['duration_min'].dropna(), bins=25, ax=axes[0, 1], color='teal', kde=True)
axes[0, 1].set_title('Movie Duration Distribution (Cleaned)')

# 3. Top 8 Primary Genres
df['primary_genre'].value_counts().head(8).plot(kind='barh', ax=axes[0, 2], color='purple')
axes[0, 2].set_title('Top 8 Primary Genres')

# 4. Content Additions by Year
df['added_year'].value_counts().sort_index().plot(kind='line', marker='o', ax=axes[1, 0], color='darkred')
axes[1, 0].set_title('Content Added to Netflix by Year')

# 5. Rating Category Distribution
df['rating'].value_counts().head(8).plot(kind='bar', ax=axes[1, 1], color='darkgreen')
axes[1, 1].set_title('Rating Category Counts')

# 6. Scatter: Release Year vs Duration
sns.scatterplot(data=df[df['type']=='Movie'], x='release_year', y='duration_min', alpha=0.4, ax=axes[1, 2], color='navy')
axes[1, 2].set_title('Release Year vs Movie Duration')

plt.tight_layout()
plt.show()


## Step 28: Final Feature Engineering Comparison
Comparison of feature engineering strategies between Content-Based Recommender Systems and Collaborative Filtering Recommender Systems.


| Step / Feature | Content-Based System | Collaborative Filtering |
| :--- | :--- | :--- |
| **Main Data Used** | Movie metadata (genre, cast, director, description) | User-item rating interaction log |
| **Text Cleaning** | Required (lowercase, punctuation, stop-words) | Not required |
| **IQR Outlier Target** | Movie duration, release year | User rating activity count, rating scores |
| **Z-Score Target** | Feature length, text token length | User/Movie mean rating metrics |
| **Categorical Encoding** | One-Hot / Label encoding for genres/tags | Integer encoding for user_id & show_id |
| **Feature Vector Format** | Sparse TF-IDF / Count Matrix ($N \times M$) | User-Movie Sparse Matrix ($U \times I$) |
| **Scaling Method** | MinMaxScaler / StandardScaler for numeric tags | Normalization of mean user ratings |
| **Main Technique** | Cosine Similarity on text embeddings | Matrix Factorization / SVD / Cosine Sim |
| **Final Recommender Input**| Text feature embeddings matrix | Sparse User-Item Pivot Table Matrix |
